# NeuroSAR: A Physics-Informed Neural Network Surrogate for SAR ADC Design

---

> *"The holy grail of analog design automation is a simulator that is both fast enough to explore the design space and accurate enough to trust."*  
> — ISSCC 2022 Short Course

---

## Welcome

This notebook series documents **NeuroSAR** — a physics-informed neural network (PINN) that learns to emulate the transient dynamics of a charge-redistribution successive-approximation register (SAR) analog-to-digital converter (ADC).  

The goal is to make the SAR ADC design space *differentiable*: once the surrogate is trained, you can compute gradients of any figure of merit (FoM) with respect to any design parameter in microseconds — enabling gradient-based inverse design, interactive exploration, and sensitivity analysis that would take hours in SPICE.

This notebook is a self-contained tutorial. No prior knowledge of machine learning is assumed, though familiarity with basic ADC concepts will help. By the end of this series, you will be able to:

1. Generate a synthetic training dataset of physically-plausible SAR waveforms.
2. Train a PINN surrogate on those waveforms.
3. Evaluate waveform fidelity and error distributions.
4. Explore the 9-dimensional design space via 2-D FoM heatmaps.
5. Run gradient-based inverse design to minimise metastability.
6. Interact with a live dashboard that updates predictions in real time.

---

## 1. What Is a SAR ADC?

A **successive-approximation register ADC** converts an analog voltage $V_{in}$ into an $N$-bit binary code by executing a binary search in $N$ steps, one bit per clock cycle. The architecture is the dominant topology in IoT, medical sensing, and wireless front-ends because it achieves excellent energy efficiency at sample rates from a few kHz to a few hundred MHz.

### 1.1 The Binary Search Algorithm

The conversion proceeds MSB-first. At step $k$:

1. The DAC outputs a trial voltage $V_{trial,k}$.
2. The comparator resolves the sign of $V_{in} - V_{trial,k}$.
3. If positive, bit $k = 1$ and the trial is kept; otherwise bit $k = 0$ and it is discarded.
4. The DAC is updated for step $k+1$.

After $N$ steps the digital code $\{b_0, b_1, \ldots, b_{N-1}\}$ approximates $V_{in}$ to within $\frac{1}{2}$ LSB.

### 1.2 Architecture Overview

```
                         ┌────────────────────────────────────────────────────┐
                         │              SAR ADC  (10-bit example)             │
                         │                                                    │
  V_in ──────┬──────────▶│  S/H ──▶  Cap-DAC Array  ──▶  Comparator ──┐     │
             │           │              (binary-                        │     │
  V_ref ─────┤           │               weighted)        ┌────────────▼──┐  │
             │           │                                │  SAR Logic    │  │
             │           │    Cu  2Cu  4Cu  ... 512Cu     │  (FSM)        │  │
             │           │    ─┬───┬────┬──────┬──        │               │  │
             │           │     │   │    │      │  ←───────│  b9...b0      │  │
             │           │    DAC NODE (V_dac)            └───────────────┘  │
             │           │        │                                           │
             └───────────│────────┘  C_load (parasitics + input cap)        │
                         │                                                    │
                         │  Digital Output: D[9:0] = b9 b8 ... b0            │
                         └────────────────────────────────────────────────────┘
```

The three sub-circuits that dominate SAR performance are:

| Sub-circuit | Key parameter | Effect on performance |
|---|---|---|
| **Capacitive DAC** | Unit cap $C_u$, load $C_{load}$ | Linearity, settling speed, switching energy |
| **Comparator** | Trans-conductance $g_m$, offset $V_{os}$ | Speed, metastability probability |
| **SAR Logic** | Clock frequency $f_s$, bit cycles | Throughput, power |


---

## 2. Physics of the SAR ADC Front-End

### 2.1 Capacitive DAC: Charge Redistribution

The DAC uses a **binary-weighted capacitor array**. The total array capacitance is:

$$C_{total} = 2^N \cdot C_u$$

During sampling, all bottom plates are connected to $V_{in}$, storing charge $Q_{init} = C_{total} \cdot V_{in}$.

At bit trial $k$ (MSB-first), capacitor $C_k = 2^{N-1-k} C_u$ is switched from GND to $V_{ref}$ (if the SAR logic drives a `1`) or from $V_{ref}$ to GND (if `0`). By charge conservation at the DAC top-plate node:

$$\Delta V_{dac,k} = \frac{C_k}{C_{total} + C_{load}} \cdot V_{ref} \cdot (2 b_k - 1)$$

The DAC node voltage after each trial is therefore:

$$V_{dac,k+1} = V_{dac,k} + \Delta V_{dac,k}$$

This is the **charge redistribution** principle. The parasitic load $C_{load}$ (routing, comparator input) attenuates the step size and is a key accuracy knob.

### 2.2 DAC Settling Dynamics

After each switching event, the DAC node settles exponentially toward its final value:

$$V(t) = V_{final} + (V_{step} - V_{final}) \cdot e^{-t / \tau_{settle}}$$

where $\tau_{settle} = R_{on} \cdot (C_{total} + C_{load})$ and $R_{on}$ is the switch resistance. Insufficient settling time before the comparator samples introduces a **dynamic non-linearity** that degrades ENOB.

### 2.3 Comparator Regeneration

The strong-arm or differential-pair latch comparator operates in two phases:

**Pre-amplification** (linear):  
$$V_{in,comp} = A_{v0} \cdot (V_{residue} + V_{os})$$
where $A_{v0}$ is the pre-amplifier gain and $V_{os}$ is the input-referred random offset.

**Regeneration** (exponential ODE):  
$$\frac{d V_{diff}}{dt} = \frac{g_m}{C_L} \cdot V_{diff}$$

This is a first-order linear ODE with solution:

$$V_{diff}(t) = V_{in,comp} \cdot \exp\!\left(\frac{g_m}{C_L} \cdot t\right)$$

The **regeneration time constant** is:

$$\tau_{regen} = \frac{C_L}{g_m}$$

The comparator resolves when $|V_{diff}|$ reaches the rail ($\approx V_{DD}$). Rail-clamping is modelled as:

$$V_{diff}(t) = \text{clip}\!\left(V_{in,comp} \cdot e^{t/\tau_{regen}},\; -V_{DD},\; +V_{DD}\right)$$

### 2.4 Metastability

**Metastability** occurs when the comparator input $V_{residue}$ is very small (near zero), so the exponential regeneration takes an unacceptably long time to cross the rail. The dwell time — defined as the time for $|V_{diff}|$ to reach a decision threshold $V_{th}$ — is:

$$t_{meta} = \tau_{regen} \cdot \ln\!\left(\frac{V_{th}}{A_{v0} \cdot |V_{residue}|}\right)$$

When $t_{meta}$ exceeds the available bit cycle time, the comparator fails to resolve, injecting a random error into the digital code — a **metastability event**. This is especially dangerous at higher sample rates where bit cycles are shorter.

The metastability probability falls exponentially with the available resolution time $T_{bit}$:

$$P_{meta} \propto \exp\!\left(-\frac{g_m}{C_L} \cdot T_{bit}\right)$$

Maximising $g_m / C_L$ (i.e., minimising $\tau_{regen}$) is the primary comparator design knob.

### 2.5 Energy Model

**DAC switching energy** (monotonic switching scheme):

$$E_{DAC} = \sum_{k=0}^{N-1} C_k \cdot V_{ref}^2 = C_{total} \cdot V_{ref}^2 \quad \text{(worst-case)}$$

**Comparator dynamic energy**:

$$E_{comp} \approx g_m \cdot V_{DD} \cdot \tau_{regen} \cdot N$$

**Total conversion energy**:
$$E_{conv} = E_{DAC} + E_{comp}$$

The **Walden Figure of Merit** normalises energy by conversion rate and resolution:

$$\text{FoM}_W = \frac{E_{conv}}{f_s \cdot 2^{\text{ENOB}}} \quad [\text{J/conv-step}]$$

State-of-the-art SAR ADCs achieve $\text{FoM}_W < 5 \text{ fJ/conv-step}$.

---

## 3. Why Physics-Informed Neural Networks?

### 3.1 The Simulation Bottleneck

Modern analog design relies on SPICE-level transient simulation for accuracy. For a 10-bit SAR ADC at 50 MHz:

- One full conversion: ~200 ns of simulation time
- A 20×20 parameter sweep: 400 simulation runs × 30 seconds each ≈ **3.3 hours**
- Gradient computation for inverse design: requires finite-difference perturbations ×9 parameters ≈ **30 hours**

This makes design-space exploration and gradient-based optimisation computationally prohibitive at the transistor level.

### 3.2 Pure Data-Driven Surrogates: Not Enough

A naive neural network trained purely on SPICE outputs can interpolate within the training data, but:

- It **extrapolates poorly** outside the training hull — exactly where inverse design ventures.
- It learns no **causal structure** — it cannot distinguish physically plausible from unphysical predictions.
- It requires **enormous datasets** to learn smooth physical relations.

### 3.3 The PINN Solution

A **physics-informed neural network** incorporates the governing equations of the system directly into the loss function. For NeuroSAR, the total training loss is:

$$\mathcal{L} = w_{data} \mathcal{L}_{data} + w_{KCL} \mathcal{L}_{KCL} + w_{charge} \mathcal{L}_{charge} + w_{ODE} \mathcal{L}_{ODE} + w_{smooth} \mathcal{L}_{smooth}$$

Each term:

| Term | Physical law enforced | Why it matters |
|---|---|---|
| $\mathcal{L}_{data}$ | Fit to reference waveforms | Accuracy |
| $\mathcal{L}_{KCL}$ | Current conservation at DAC node | Prevents unphysical charge flows |
| $\mathcal{L}_{charge}$ | Charge conservation across switching | Ensures DAC steps are exact |
| $\mathcal{L}_{ODE}$ | Comparator regeneration ODE | Correct exponential dynamics |
| $\mathcal{L}_{smooth}$ | Waveform curvature penalty | Suppresses high-freq artefacts |

By embedding physics into the loss, the network is constrained to produce *physically realizable* waveforms even at design points it has never seen during training. This dramatically improves extrapolation and data efficiency.

### 3.4 The Differentiable Surrogate Advantage

Once trained, the PINN is a differentiable function:

$$f_{\theta}: (V_{in}, V_{ref}, C_u, C_{load}, g_m, \tau, V_{os}, T, f_s) \rightarrow (V_{dac}(t), V_{diff}(t), V_{comp}(t), E_{conv})$$

Gradients $\partial \text{FoM} / \partial p_i$ are computed by **automatic differentiation** in microseconds, enabling:

- **Sensitivity analysis**: which parameter matters most for metastability?
- **Gradient-based inverse design**: find $\mathbf{p}^*$ that minimises metastability subject to energy budget.
- **Interactive exploration**: sliders update waveforms at 60 fps.

This is fundamentally impossible with SPICE, which is a black-box non-differentiable simulator.

---

## 4. Project Structure

```
NeuroSAR/
├── src/
│   ├── config.py          — Design space bounds, training hyper-parameters, paths
│   ├── utils.py           — Seeding, device selection, tensor helpers
│   ├── physics.py         — Analytical SAR physics (auto-differentiable, PyTorch)
│   ├── dataset.py         — Mode A (synthetic) and Mode B (SPICE CSV) data pipelines
│   ├── simulate_spice.py  — ngspice / Xyce deck templates and parsers
│   ├── pinn_model.py      — NeuroSARNet architecture (Fourier features + MLP heads)
│   ├── losses.py          — All five PINN loss terms
│   ├── train_pinn.py      — Training loop, scheduler, checkpointing
│   ├── evaluate.py        — Inference, batch metrics, CSV export
│   ├── fom_analysis.py    — 1-D and 2-D parameter sweeps, FoM surfaces
│   ├── inverse_design.py  — Gradient-based design optimisation
│   ├── plotting.py        — Plotly figures for waveforms, heatmaps, trajectories
│   ├── animation.py       — Animated bit-cycle and metastability visualisations
│   ├── interactive_ui.py  — ipywidgets dashboard (InteractiveDashboard)
│   └── export_results.py  — Artefact packaging, model cards, CSV export
├── notebooks/
│   ├── 00_Project_Overview.ipynb       ← You are here
│   ├── 01_Generate_Dataset.ipynb
│   ├── 02_Train_PINN.ipynb
│   ├── 03_Evaluate_Waveforms.ipynb
│   ├── 04_FoM_Explorer.ipynb
│   ├── 05_Inverse_Design.ipynb
│   └── 06_NeuroSAR_Interactive_Demo.ipynb
├── data/
│   ├── raw/               — Raw SPICE transient outputs
│   ├── processed/         — Generated datasets (*.pt)
│   ├── checkpoints/       — Model checkpoints (*.pt)
│   └── exports/           — CSV predictions, sweep data
└── assets/
    └── figures/           — Exported PNG / HTML figures
```

---

## 5. Design Space

NeuroSAR models a **9-dimensional design space** covering the dominant parameters of the SAR ADC front-end:

| Parameter | Symbol | Range | Unit | Physical role |
|---|---|---|---|---|
| Input voltage | $V_{in}$ | [0.0, 1.8] | V | Conversion point |
| Reference voltage | $V_{ref}$ | 1.8 (fixed) | V | Sky130 nominal supply |
| Unit capacitor | $C_u$ | [1, 50] | fF | DAC resolution, energy |
| Load capacitance | $C_{load}$ | [10, 500] | fF | Settling speed, linearity |
| Transconductance | $g_m$ | [50, 2000] | µS | Comparator speed |
| Regen time const. | $\tau_{regen}$ | [10, 500] | ps | Metastability threshold |
| Input offset | $V_{os}$ | [−10, +10] | mV | Systematic error |
| Temperature | $T$ | [250, 400] | K | Process corner |
| Sample rate | $f_s$ | [1, 200] | MHz | Throughput |

The parameter space is sampled uniformly at random during dataset generation, covering all corners simultaneously.

### 5.1 Mode A vs Mode B Dataset

**Mode A — Synthetic (default):**  
Waveforms are generated analytically using the physics models in `src/physics.py`. No external tools are needed. This mode enables rapid iteration and is the primary training source for NeuroSAR.

**Mode B — Real SPICE:**  
For production use, `src/simulate_spice.py` generates ngspice or Xyce netlists for an open-source Sky130 SAR ADC testbench. The resulting transient CSV or HDF5 files are loaded via `dataset.load_spice_csv()` or `dataset.load_spice_hdf5()`. Mode B data can be blended with Mode A to combine physical accuracy with broad coverage.

---

## 6. The NeuroSARNet Architecture

```
  Design Params (9)  ──────────────────────────────────────────┐
  Bit Index    (1)  ──────────────────────────────────────────▶│  Concatenate
  Time Fourier (64) ◀── FourierFeatures(t_local) ─────────────┘       │
                                                                        │
                                                                        ▼
                                                          ┌────────────────────────┐
                                                          │    Shared Trunk MLP    │
                                                          │  Linear(in→256) + Tanh │
                                                          │  Linear(256→256) + Tanh│
                                                          │  Linear(256→256) + Tanh│
                                                          │  Linear(256→128) + Tanh│
                                                          └────────────┬───────────┘
                                                                       │
                          ┌────────────────┬─────────────┬────────────┤
                          ▼                ▼             ▼            ▼
                   ┌─────────────┐ ┌──────────────┐ ┌──────────┐ ┌──────────┐
                   │ Head: vdac  │ │ Head: vdiff  │ │Head:vcomp│ │Head:energy│
                   │ Linear→64  │ │ Linear→128   │ │Linear→128│ │Linear→64 │
                   │ Tanh→1     │ │ Tanh→T       │ │Tanh→T    │ │Softplus→1│
                   └─────────────┘ └──────────────┘ └──────────┘ └──────────┘
                       V_dac(1)       V_diff(T)       V_comp(T)    Energy(1)
```

**Key architectural choices:**

- **Tanh activations** provide smooth second derivatives, which are essential for computing the physics residuals (the ODE loss requires $dV_{comp}/dt$).
- **Fourier feature encoding** of the time axis maps $t \in [0,1]$ to $[\sin(2\pi \sigma_i t), \cos(2\pi \sigma_i t)]$ for a bank of 32 frequencies. This spectral lifting dramatically accelerates learning of the exponential transient behaviour, following Tancik et al. (2020).
- **Separate output heads** allow each physical quantity to have its own expressivity budget and enforce head-specific loss terms without coupling.
- **Softplus output on energy** guarantees the predicted energy is always positive — a hard physical constraint.

Total trainable parameters: **~430,000**. At inference time, a single design point across all 10 bit trials takes **< 1 ms** on CPU.

---

## 7. PINN Loss Functions: Physics as Regularisation

The five loss terms are detailed in `src/losses.py`. Here we explain the physical meaning of each.

### 7.1 Data Loss $\mathcal{L}_{data}$

$$\mathcal{L}_{data} = \text{MSE}(\hat{V}_{dac}, V_{dac}) + \text{MSE}(\hat{V}_{diff}, V_{diff}) + \text{MSE}(\hat{V}_{comp}, V_{comp}) + \text{MSE}(\hat{E}, E)$$

This is the standard supervised regression loss, fitting the model to reference waveforms.

### 7.2 KCL Residual $\mathcal{L}_{KCL}$

At the DAC top-plate node, Kirchhoff's Current Law requires that after switching, the settling transient obeys:

$$(C_{total} + C_{load}) \frac{d V_{dac}}{dt} = 0 \quad \text{(no driving current after switch)}$$

We enforce this by penalising the time derivative of $V_{diff}$ in the later portion of each bit window (where the DAC should have settled), weighted by a ramp $w(t) \in [0.2, 1.0]$:

$$\mathcal{L}_{KCL} = \left\| (C_{total} + C_{load}) \cdot \frac{d \hat{V}_{diff}}{dt} \cdot w(t) \right\|_2^2$$

### 7.3 Charge Conservation $\mathcal{L}_{charge}$

Each switching event must conserve charge. The predicted DAC voltage step at bit $k$ must match the physics model:

$$\mathcal{L}_{charge} = \frac{1}{N-1} \sum_{k=0}^{N-2} \left( \underbrace{\hat{V}_{dac,k+1} - \hat{V}_{dac,k}}_{\text{predicted step}} - \underbrace{\frac{C_k}{C_{total} + C_{load}} V_{ref} (2b_k - 1)}_{\text{physics step}} \right)^2$$

### 7.4 Comparator ODE Residual $\mathcal{L}_{ODE}$

The comparator output must satisfy the regeneration ODE in the linear region ($|V_{comp}| < 0.95 V_{DD}$):

$$\mathcal{L}_{ODE} = \left\| \mathbf{1}_{|V_{comp}| < 0.95 V_{DD}} \cdot \left( \frac{d \hat{V}_{comp}}{dt} - \frac{g_m}{C_L} \hat{V}_{comp} \right) \right\|_2^2$$

The mask excludes the rail-limited region where the linear ODE no longer holds.

### 7.5 Smoothness Loss $\mathcal{L}_{smooth}$

A second-order finite-difference penalty discourages physically implausible oscillations:

$$\mathcal{L}_{smooth} = \| \Delta^2 \hat{V}_{diff} \|_2^2 + \| \Delta^2 \hat{V}_{comp} \|_2^2$$

where $\Delta^2 f_i = f_{i+2} - 2f_{i+1} + f_i$ is the second difference operator.

---

## 8. Notebook Roadmap

Work through the notebooks in order for the full tutorial experience, or jump to the topic you care about:

| Notebook | Topic | Key functions |
|---|---|---|
| **00** | *This overview* | — |
| **01** | Dataset generation & visualisation | `generate_synthetic_dataset`, `save_dataset`, `plot_conversion_summary` |
| **02** | PINN training | `build_dataloaders`, `NeuroSARNet`, `train`, `plot_training_history` |
| **03** | Evaluation & waveform comparison | `load_model`, `evaluate_dataset`, `export_predictions` |
| **04** | FoM design-space exploration | `parameter_sweep_2d`, `plot_fom_heatmap`, `export_sweep` |
| **05** | Gradient-based inverse design | `run_inverse_design`, `plot_inverse_trajectory` |
| **06** | Interactive dashboard | `InteractiveDashboard`, `animate_conversion`, `animate_metastability` |

---

## 9. References

1. Kuo, C.-H. et al. "A 10-bit 50-MS/s SAR ADC With a Monotonic Capacitor Switching Procedure." *JSSC* 2010.
2. Liu, C.-C. et al. "A 0.92-mW 10-bit 50-MS/s SAR ADC in 0.13-µm CMOS." *JSSC* 2010.
3. Raissi, M., Perdikaris, P., Karniadakis, G.E. "Physics-informed neural networks." *JCP* 2019.
4. Tancik, M. et al. "Fourier Features Let Networks Learn High-Frequency Functions in Low Dimensional Domains." *NeurIPS* 2020.
5. Razavi, B. *Design of Analog CMOS Integrated Circuits*, 2nd ed. McGraw-Hill, 2015.
6. Walden, R.H. "Analog-to-digital converter survey and analysis." *JSAC* 1999.

---

*Ready to begin? Open `01_Generate_Dataset.ipynb` to generate the training data.*